# LLM Instruction Following and Fine-Tuning Project

## Import All Libraries

In [ ]:
import gc
import os
import json
import glob
import torch
import warnings
import subprocess
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from datasets import load_dataset
from huggingface_hub import login
from IPython.display import display, HTML
from peft import LoraConfig, get_peft_model, PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, DataCollatorForLanguageModeling



load_dotenv()
login(os.getenv("HUGGINGFACE_TOKEN"))
warnings.filterwarnings("ignore")

## Section 1: Dataset Loading and Tokenizer Analysis

This section covers loading the instruction-following dataset and analyzing how different tokenizers process English and Persian texts.

In [ ]:
dataset_name = "allenai/tulu-3-sft-personas-instruction-following"
sample_size = 2000

print(f"\033[1mLoading dataset:\033[0m {dataset_name}")
dataset = load_dataset(dataset_name, split="train")

print(f"\033[1mNumber of Data in Dataset:\033[0m {len(dataset)}")
print(f"\033[1mDataset Features:\033[0m {dataset.features}")
selected_dataset = dataset.select(range(sample_size))
print(f"\033[1mNumber Selected Data:\033[0m {len(selected_dataset)}")

print("\n\033[1mSample Data Structure:\033[0m")
for key, value in dataset[0].items():
    if isinstance(value, list):
        print(f"\033[1m{key}:\033[0m")
        for item in value:
            print(f"  - {item}")
    else:
        print(f"\033[1m{key}:\033[0m {value}")

Loading dataset: allenai/tulu-3-sft-personas-instruction-following
Number of Data in Dataset: 29980
Dataset Features: {'id': Value('string'), 'prompt': Value('string'), 'messages': List({'content': Value('string'), 'role': Value('string')}), 'constraints': List(Value('string'))}
Number Selected Data: 2000

Sample Data Structure:
id: personas_IF_1s84skm7dqifreqgvsgzyxdf
prompt: Provide a detailed analysis of Candace Parker's defensive techniques in her recent games, excluding the words "aggressive" and "blocking", in the format of a sports commentary script.
messages:
  - {'content': 'Provide a detailed analysis of Candace Parker\'s defensive techniques in her recent games, excluding the words "aggressive" and "blocking", in the format of a sports commentary script.', 'role': 'user'}
  - {'content': "[Sports Commentary Script]\n\n[Opening Scene: A packed basketball arena, with fans eagerly awaiting the analysis of Candace Parker’s recent performances on the court.]\n\nCommentator 1: Wel

In [ ]:
TOKEN_BG_COLORS = [
    "#58a6ff", "#f78166", "#85e89d", "#ffab70",
    "#bca0ff", "#79c0ff", "#ffb4a2", "#d0d0d0",
]

def show_raw_tokens(model_name, token_list):
    html_parts = [
        '<div style="margin:20px 0; padding:15px; background:#1e1e1e; border-radius:10px; '
        'border-left:6px solid #6272a4; font-family: monospace;">'
        f'<strong style="color:#ff79c6; font-size:18px;">{model_name}</strong><br><br>'
        '<div style="line-height: 1.6; white-space: pre-wrap;">'
    ]

    for i, token in enumerate(token_list):
        color = TOKEN_BG_COLORS[i % len(TOKEN_BG_COLORS)]
        style = (
            f"background:{color}; color:black; padding:2px 2px; margin:2px; "
            f"border-radius:2px; display:inline-block; font-size:13px;"
        )

        # Only escape < and > to prevent HTML breaking
        safe_token = token.replace("<", "&lt;").replace(">", "&gt;")

        html_parts.append(f'<span style="{style}">{safe_token}</span>')

    html_parts.append('</div></div>')
    display(HTML("".join(html_parts)))

In [ ]:
sample_text1 = """English and CAPITALIZATION 🎵 漢字
show_tokens False None elif == >= else:
two tabs:"  " Three tabs: "         "
3.25 * 12 = 39"""

sample_text2 = "زبان فارسی یک زبان بسیار غنی و زیبا است که قدمتی چند هزار ساله دارد و میراث فرهنگی ارزشمندی محسوب می‌شود."

model_names = [
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "google/gemma-2-2b-it",
    "google/gemma-3-4b-it",
    "meta-llama/Llama-3.1-8B"
]

for model_name in model_names:
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name)

        tokens1 = [tokenizer.decode([t]) for t in tokenizer(sample_text1, add_special_tokens=False)["input_ids"]]
        show_raw_tokens(f"{model_name} - Sample Text 1 (English)", tokens1)

        tokens2 = [tokenizer.decode([t]) for t in tokenizer(sample_text2, add_special_tokens=False)["input_ids"]]
        show_raw_tokens(f"{model_name} - Sample Text 2 (Persian)", tokens2)

    except Exception as e:
        print(f"Error loading {model_name}: {e}")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

## Model Loading and Precision Analysis

This part involves loading the base model, testing memory consumption across various data types, and inspecting chat template formatting.

In [ ]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)
dtypes = [("float32", torch.float32), ("float16", torch.float16), ("bfloat16", torch.bfloat16)]

print(f"Model \033[1m{model_name}\033[0m loaded successfully.")
print("\nTesting different data types:")
for dtype_name, dtype in dtypes:
    try:
        test_model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=dtype, device_map="auto", trust_remote_code=True)
        param_size = sum(p.numel() for p in test_model.parameters())
        if dtype == torch.float32:
            memory = param_size * 4 / (1024**3)
        else:
            memory = param_size * 2 / (1024**3)

        print(f"{dtype_name} Memory: ≈ \033[1m{memory:.2f}\033[0m GB")
        del test_model
        gc.collect()
        torch.cuda.empty_cache()

    except Exception as e:
        print(f"Error in {dtype_name}: {str(e)}")

Model TinyLlama/TinyLlama-1.1B-Chat-v1.0 loaded successfully.

Testing different data types:


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

float32 Memory: ≈ 4.10 GB
float16 Memory: ≈ 2.05 GB
bfloat16 Memory: ≈ 2.05 GB


In [ ]:
messages = [
    {"role": "system", "content": "You are chatbot assistant in law."},
    {"role": "user", "content": "Hello, how are you?"}
]
formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False)
print("\033[1mRaw Chat Template Output:\033[0m")
print(formatted_prompt)

Raw Chat Template Output:
<|system|>
You are chatbot assistant in law.</s>
<|user|>
Hello, how are you?</s>



## Section 2: Data Preparation and LoRA Configuration

Preparing the dataset for Supervised Fine-Tuning (SFT) and configuring Parameter-Efficient Fine-Tuning (LoRA) hyperparameters.

In [ ]:
MAX_SEQ_LENGTH = 1024

def tokenize_function(examples):
    formatted_text = tokenizer.apply_chat_template(examples["messages"], tokenize=False, add_generation_prompt=False)
    return tokenizer(
        formatted_text,
        truncation=True,
        padding="max_length",
        max_length=MAX_SEQ_LENGTH
    )

processed_dataset = selected_dataset.map(tokenize_function, batched=True)

print("First entry of the processed dataset features:")
print(processed_dataset.features)

print("\nFirst entry of the tokenized input_ids:")
print(processed_dataset[0]["input_ids"])

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

First entry of the processed dataset features:
{'id': Value('string'), 'prompt': Value('string'), 'messages': List({'content': Value('string'), 'role': Value('string')}), 'constraints': List(Value('string')), 'input_ids': List(Value('int32')), 'attention_mask': List(Value('int8'))}

First entry of the tokenized input_ids:
[1, 529, 29989, 1792, 29989, 29958, 13, 1184, 29894, 680, 263, 13173, 7418, 310, 28433, 815, 24239, 29915, 29879, 822, 6270, 13698, 297, 902, 7786, 8090, 29892, 429, 22368, 278, 3838, 376, 351, 3663, 573, 29908, 322, 376, 1271, 292, 613, 297, 278, 3402, 310, 263, 14717, 3440, 653, 2471, 29889, 2, 29871, 13, 29966, 29989, 465, 22137, 29989, 29958, 13, 29961, 29903, 4011, 461, 653, 14415, 29962, 13, 13, 29961, 6585, 292, 2522, 1600, 29901, 319, 4870, 287, 20305, 564, 2386, 29892, 411, 24909, 19888, 368, 7272, 292, 278, 7418, 310, 28433, 815, 24239, 30010, 29879, 7786, 21637, 373, 278, 8973, 5586, 13, 13, 20001, 1061, 29871, 29896, 29901, 21829, 1250, 29892, 20305, 24909

## LoRA Configuration and Setup

Defining PEFT LoRA configuration parameters and loading the model with optimized memory mapping.

In [ ]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"]
)

model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True)

model = get_peft_model(model, peft_config)
print("Numbers of trainable parameters:")
model.print_trainable_parameters()

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Numbers of trainable parameters:
trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079


In [ ]:
training_args = TrainingArguments(
    output_dir="./tinyllama-sft",
    num_train_epochs=5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=50,
    weight_decay=0.01,
    fp16=True,
    save_strategy="epoch",
    eval_strategy="no",
    optim="adamw_torch",
    report_to="none"
)

## Section 3: Model Training and Weight Merging

Executing supervised fine-tuning and merging trained LoRA adapters back into the base model.

In [ ]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=processed_dataset,
    data_collator=data_collator,
)

print("Starting training...")
trainer.train()
print("Training finished.")

The model is already on multiple devices. Skipping the move to device specified in `args`.


Starting training...


Step,Training Loss
50,1.325700
100,1.273700
150,1.249300
200,1.199700
250,1.212000
300,1.175100
350,1.163500
400,1.137500
450,1.142000
500,1.130200


Training finished.


In [ ]:
finetune_model_path = "./tinyllama-finetuned"
trainer.save_model(finetune_model_path)
print(f"Fine-tuned adapters saved to {finetune_model_path}.")

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
base_model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")

model_to_merge = PeftModel.from_pretrained(base_model, finetune_model_path)
merged_model = model_to_merge.merge_and_unload()

merged_model_path = "./tinyllama-merged"
merged_model.save_pretrained(merged_model_path)
tokenizer.save_pretrained(merged_model_path)
print(f"Merged adaters saved to {merged_model_path}")

def get_directory_size(dir_path):
    return sum(
        os.path.getsize(os.path.join(dir_path, f))
        for f in os.listdir(dir_path)
        if os.path.isfile(os.path.join(dir_path, f))
    )

adapter_size = get_directory_size(finetune_model_path)
merged_size = get_directory_size(merged_model_path)

print(f"Adapter Size: {adapter_size / 1024**2:.2f} MB")
print(f"Merged Model Size: {merged_size / 1024**3:.2f} GB")

Fine-tuned adapters saved to ./tinyllama-finetuned.
Merged adaters saved to ./tinyllama-merged
Adapter Size: 21.15 MB
Merged Model Size: 2.05 GB


## Section 4: Model Evaluation

Performing qualitative evaluation on prompt responses and quantitative assessment using the IFEval benchmark.

In [ ]:
prompts = [
    "Create a slogan for a new vintage blues record collection app.",
    "Create a slogan for a new vintage blues record collection app. The slogan should written in all capital letters.",
    "Create a slogan for a new vintage blues record collection app. The slogan should written in all capital letters and end with the word soul.",
    "Create a slogan for a new vintage blues record collection app. The slogan should written in all capital letters, end with the word soul, and contain exactly seven words.",
    "Create a slogan for a new vintage blues record collection app. The slogan should written in all capital letters, end with the word soul, and contain exactly seven words. Wrap the entire answer in JSON format."
]

device = "cuda" if torch.cuda.is_available() else "cpu"

def generate_response(model, tokenizer, prompt):
    messages = [{"role": "user", "content": prompt}]
    input_ids = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True
    ).to(device)
    with torch.no_grad():
        outputs = model.generate(input_ids, max_new_tokens=100, do_sample=True, temperature=0.6, repetition_penalty=1.2, pad_token_id=tokenizer.eos_token_id)

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = response.split("assistant\n")[-1] if "assistant\n" in response else response
    response = response.strip()
    return response

def evaluate_model(model_path, prompts, model_name_display):
    print(f"\nLoading {model_name_display} model from {model_path}")
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto", torch_dtype=torch.float16)

    print(" - Qualitative Evaluation Results:\n")
    for i, p in enumerate(prompts):
        print(f"  - Prompt {i+1}: {p}")
        response = generate_response(model, tokenizer, p)
        print(f"  - Response: {response}\n")
        # print("-"*30)

    del model
    torch.cuda.empty_cache()
    gc.collect()

evaluate_model(finetune_model_path, prompts, "Fine-Tuned")
evaluate_model(model_name, prompts, "Base")


Loading Fine-Tuned model from ./tinyllama-finetuned
 - Qualitative Evaluation Results:

  - Prompt 1: Create a slogan for a new vintage blues record collection app.
  - Response: <|user|>
Create a slogan for a new vintage blues record collection app. 
<|assistant|>
"Discover the Blues that Move You Away from the Mundane, with Each Song as Your Guide."

This slogan encourages users to discover and immerse themselves in the history of blues music through a curated selection of classic tracks and exclusive live performances by contemporary artists. The emphasis on "moving you away from the mundane" is intended to appeal to those who appreciate authenticity and passionate expression in their music listening experiences. Additionally, it

  - Prompt 2: Create a slogan for a new vintage blues record collection app. The slogan should written in all capital letters.
  - Response: <|user|>
Create a slogan for a new vintage blues record collection app. The slogan should written in all capital l

In [10]:
merged_model_path = "./tinyllama-merged"
merged_output_dir = Path("./lm_eval_outputs_merged")
merged_log_file = Path("evaluation_logs_merged.txt")

merged_output_dir.mkdir(parents=True, exist_ok=True)

cmd = [
    "lm_eval",
    "--model", "hf",
    "--model_args", f"pretrained={merged_model_path},dtype=float16",
    "--tasks", "ifeval",
    "--batch_size", "auto",
    "--output_path", str(merged_output_dir),
]

with open(merged_log_file, "w", encoding="utf-8") as lf:
    proc = subprocess.run(cmd, stdout=lf, stderr=subprocess.STDOUT)

if proc.returncode == 0:
    print("✅ Evaluation finished successfully!")
    print(f"Outputs are in: {merged_output_dir.resolve()}")
    print(f"Full logs saved to: {merged_log_file.resolve()}")
else:
    print("❌ Evaluation failed. Check logs:", merged_log_file.resolve())


✅ Evaluation finished successfully!
Outputs are in: /content/lm_eval_outputs_merged
Full logs saved to: /content/evaluation_logs_merged.txt


In [12]:
based_model_path = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
base_output_dir = Path("./lm_eval_outputs_base")
base_log_file = Path("evaluation_logs_base.txt")

base_output_dir.mkdir(parents=True, exist_ok=True)

cmd = [
    "lm_eval",
    "--model", "hf",
    "--model_args", f"pretrained={based_model_path},dtype=float16",
    "--tasks", "ifeval",
    "--batch_size", "auto",
    "--output_path", str(base_output_dir),
]

with open(base_log_file, "w", encoding="utf-8") as lf:
    proc = subprocess.run(cmd, stdout=lf, stderr=subprocess.STDOUT)

if proc.returncode == 0:
    print("✅ Evaluation finished successfully!")
    print(f"Outputs are in: {base_output_dir.resolve()}")
    print(f"Full logs saved to: {base_log_file.resolve()}")
else:
    print("❌ Evaluation failed. Check logs:", base_log_file.resolve())


✅ Evaluation finished successfully!
Outputs are in: /content/lm_eval_outputs_base
Full logs saved to: /content/evaluation_logs_base.txt


In [9]:
def extract_ifeval_results(folder_path):
    json_files = glob.glob(os.path.join(folder_path, "**/*.json"), recursive=True)
    results_file = next((f for f in json_files if "results" in f), None) or (json_files[0] if json_files else None)

    if not results_file:
        print(f"⚠️ Warning: No file found in folder {folder_path}.")
        return None

    with open(results_file, 'r') as f:
        data = json.load(f)

    results = data.get('results', {}).get('ifeval', {})

    def get_val(key):
        return results.get(f"{key},none", results.get(key, 0)) * 100

    return {
        "Prompt Level Strict Acc": get_val("prompt_level_strict_acc"),
        "Inst Level Strict Acc": get_val("inst_level_strict_acc"),
        "Prompt Level Loose Acc": get_val("prompt_level_loose_acc"),
        "Inst Level Loose Acc": get_val("inst_level_loose_acc")
    }

def create_comparison_table(base_folder, ft_folder):
    base_data = extract_ifeval_results(base_folder)
    ft_data = extract_ifeval_results(ft_folder)

    if not base_data or not ft_data:
        return "Error in reading data."

    metrics_list = [
        "Inst Level Strict Acc",
        "Inst Level Loose Acc",
        "Prompt Level Loose Acc",
        "Prompt Level Strict Acc"
    ]

    table_rows = []

    for m in metrics_list:
        base_val = base_data[m]
        ft_val = ft_data[m]
        improvement = ft_val - base_val
        growth = ft_val / base_val if base_val > 0 else float('inf')

        table_rows.append({
            "Metric": m,
            "Base Model": f"{base_val:.2f}%",
            "Fine-Tuned Model": f"{ft_val:.2f}%",
            "Improvement": f"{improvement:+.2f}%"
        })

    df = pd.DataFrame(table_rows)
    print("\n" + "-"*10 + " IFEval Final Evaluation Comparison Table " + "-"*10)
    display(df)

    return df

output_dir_base = "./results_base/"
output_dir_ft = "./results_finetuned/"

final_df = create_comparison_table(output_dir_base, output_dir_ft)


---------- IFEval Final Evaluation Comparison Table ----------


,Metric,Base Model,Fine-Tuned Model,Enhancement
0,Inst Level Strict Acc,12.55%,34.65%,+22.10%
1,Inst Level Loose Acc,13.15%,36.45%,+23.30%
2,Prompt Level Loose Acc,9.25%,22.55%,+13.30%
3,Prompt Level Strict Acc,8.51%,20.33%,+11.82%
